# 02 - Risk Metrics and Portfolio Construction

**Business question:** which coins gave the best return *per unit of risk*, how bad can losses get, and does optimising a portfolio actually help out-of-sample?

Sections: risk table -> drawdowns -> Value-at-Risk (and why Gaussian VaR fails) -> efficient frontier -> **out-of-sample test of optimisation** -> risk contributions -> your own portfolio (XIRR + FIFO).

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "analytics").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from analytics import data, metrics, stats_tests, forecasting
from analytics.portfolio import xirr, fifo_realized_pnl

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4.5)
pd.options.display.float_format = "{:,.4f}".format

In [ ]:
ohlcv = data.load_ohlcv()
prices = ohlcv.pivot(index="date", columns="symbol", values="close").sort_index()
volume = ohlcv.pivot(index="date", columns="symbol", values="volume").sort_index()
sources = data.data_sources()
print("Data source(s):", sources, "|", prices.shape[1], "coins |", prices.index.min().date(), "->", prices.index.max().date())
if "synthetic" in sources:
    print("\n*** WARNING: SYNTHETIC DEMO DATA. Re-run `python -m analytics.etl.ingest` for real data before reading anything into the results. ***")

## Risk and return by coin

In [ ]:
simple = metrics.simple_returns(prices)
table = metrics.risk_table(prices)
fmt = {c: "{:.1%}" for c in ["total_return", "ann_return", "ann_volatility", "max_drawdown", "var_95", "cvar_95"]}
display(table.sort_values("sharpe", ascending=False).style.format({**fmt, "sharpe": "{:.2f}", "sortino": "{:.2f}", "calmar": "{:.2f}", "skew": "{:.2f}", "excess_kurtosis": "{:.1f}", "beta_vs_btc": "{:.2f}"}))

## Drawdowns - how far, how long?

In [ ]:
dd = pd.DataFrame({s: metrics.drawdown_series(prices[s].dropna()) for s in prices.columns})
dd[[c for c in ["BTC", "ETH", "SOL", "ADA"] if c in dd.columns]].plot(title="Drawdown from running peak"); plt.show()

events = pd.DataFrame({s: metrics.max_drawdown(prices[s].dropna()) for s in prices.columns}).T
events["days_to_recover"] = [(r - t).days if r is not None and pd.notna(r) else np.nan for r, t in zip(events["recovery_date"], events["trough_date"])]
events["days_peak_to_trough"] = [(t - p).days for p, t in zip(events["peak_date"], events["trough_date"])]
display(events)

## Value-at-Risk: historical vs Gaussian

In [ ]:
rows = {}
for s in simple.columns:
    r = simple[s].dropna()
    gaussian_var = -(r.mean() + stats.norm.ppf(0.01) * r.std())
    rows[s] = {"historical_VaR_99": metrics.historical_var(r, 0.99), "gaussian_VaR_99": gaussian_var,
               "CVaR_99": metrics.historical_cvar(r, 0.99), "exceed_gaussian_VaR_share": (r < -gaussian_var).mean()}
var_cmp = pd.DataFrame(rows).T
display(var_cmp.style.format("{:.2%}"))
var_cmp[["historical_VaR_99", "gaussian_VaR_99"]].plot.bar(title="1-day 99% VaR: what actually happened vs the Gaussian assumption"); plt.show()
print("If the Gaussian model were right, losses would exceed its 99% VaR on 1.0% of days.")

> **Interpretation:** *(does the Gaussian model underestimate tail risk? by how much? which coins are worst?)*

## Efficient frontier

In [ ]:
universe = [c for c in ["BTC", "ETH", "SOL", "BNB", "XRP", "ADA"] if c in simple.columns] or list(simple.columns[:6])
rets = simple[universe].dropna()
CAP = 0.5   # no single coin above 50%
frontier = metrics.efficient_frontier(rets, n_points=30, max_weight=CAP)
w_min = metrics.min_variance_weights(rets, CAP)
w_sharpe = metrics.max_sharpe_weights(rets, max_weight=CAP)
w_equal = pd.Series(1 / len(universe), index=universe)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(frontier["volatility"], frontier["return"], "-o", ms=3, label="efficient frontier")
ax.scatter([metrics.annualized_vol(rets[s]) for s in universe], rets.mean() * metrics.PERIODS, c="grey", label="single coins")
for s in universe:
    ax.annotate(s, (metrics.annualized_vol(rets[s]), rets[s].mean() * metrics.PERIODS))
for name, w, colour in [("min variance", w_min, "green"), ("max Sharpe", w_sharpe, "red"), ("equal weight", w_equal, "orange")]:
    st_ = metrics.portfolio_stats(w, rets)
    ax.scatter(st_["volatility"], st_["return"], s=140, c=colour, marker="*", label=name, zorder=5)
ax.set(xlabel="annualised volatility", ylabel="annualised return (arithmetic)", title=f"Long-only frontier, max weight {CAP:.0%}  (IN-SAMPLE)")
ax.legend(); plt.show()
display(pd.DataFrame({"min_variance": w_min, "max_sharpe": w_sharpe, "equal_weight": w_equal}).style.format("{:.1%}"))

## The honest test: does optimisation survive out-of-sample?

In [ ]:
split = int(len(rets) * 0.7)
train, test = rets.iloc[:split], rets.iloc[split:]
candidates = {
    "equal_weight": pd.Series(1 / len(universe), index=universe),
    "min_variance": metrics.min_variance_weights(train, CAP),
    "max_sharpe": metrics.max_sharpe_weights(train, max_weight=CAP),
}
oos = pd.DataFrame({name: {
    "in_sample_sharpe": metrics.portfolio_stats(w, train)["sharpe"],
    "out_of_sample_sharpe": metrics.portfolio_stats(w, test)["sharpe"],
    "out_of_sample_vol": metrics.portfolio_stats(w, test)["volatility"],
} for name, w in candidates.items()}).T
display(oos.style.format("{:.2f}"))
oos[["in_sample_sharpe", "out_of_sample_sharpe"]].plot.bar(title="Sharpe: in-sample vs out-of-sample (weights fit on the first 70% of days)"); plt.show()

> **Interpretation:** *(a big gap between the two bars is overfitting. Did max-Sharpe hold up, or did simple equal weighting do as well?)*

## Where does the risk actually come from?

In [ ]:
contrib = metrics.risk_contributions(w_equal, rets)
pd.DataFrame({"weight": w_equal, "share_of_portfolio_risk": contrib}).plot.bar(title="Equal-weight portfolio: capital share vs risk share")
plt.show()
div_ratio = metrics.diversification_ratio(w_equal, rets)
print(f"Diversification ratio (equal weight): {div_ratio:.2f}  (1.00 = no benefit from diversifying)")

## Your portfolio: money-weighted return and FIFO realised P&L

In [ ]:
USER_ID = 1
portfolio_ok = False
try:
    value = data.load_portfolio_value(USER_ID)
    tx = data.load_transactions(USER_ID)
    flows = pd.DataFrame({"date": tx["date"], "amount": np.where(tx["type"] == "buy", -1, 1) * tx["quantity"] * tx["price"]})
    flows = pd.concat([flows, pd.DataFrame({"date": [value.index[-1]], "amount": [value["value"].iloc[-1]]})])
    irr = xirr(flows["date"], flows["amount"])
    realised, open_lots = fifo_realized_pnl(tx)
    value["value"].plot(title="Portfolio value rebuilt from the transaction ledger (SQL view)"); plt.show()
    print(f"XIRR: {irr:.1%}   |   realised P&L (FIFO): {realised['realized_pnl'].sum() if len(realised) else 0:,.2f}")
    display(realised); display(open_lots)
    portfolio_ok = True
except Exception as error:
    print("Portfolio section skipped:", error)

## Key takeaways (computed)

In [ ]:
best = table["sharpe"].idxmax(); worst_dd = table["max_drawdown"].idxmin()
lines = [
    f"- **Best risk-adjusted return:** {best} (Sharpe {table.loc[best, 'sharpe']:.2f}); deepest drawdown: {worst_dd} ({table.loc[worst_dd, 'max_drawdown']:.0%}).",
    f"- **Tail risk:** across coins the Gaussian 99% VaR was exceeded on {var_cmp['exceed_gaussian_VaR_share'].mean():.1%} of days on average (1.0% if the Gaussian were right).",
    f"- **Optimisation out-of-sample:** max-Sharpe Sharpe fell from {oos.loc['max_sharpe', 'in_sample_sharpe']:.2f} (in-sample) to {oos.loc['max_sharpe', 'out_of_sample_sharpe']:.2f} (out-of-sample); equal weight was {oos.loc['equal_weight', 'out_of_sample_sharpe']:.2f}.",
    f"- **Diversification:** ratio {div_ratio:.2f} for an equal-weight basket of {len(universe)} coins.",
]
if portfolio_ok:
    lines.append(f"- **Your portfolio:** XIRR {irr:.1%}.")
from IPython.display import Markdown
Markdown("\n".join(lines))